In [1]:
import boto3, os
from io import StringIO
import pandas as pd
from dotenv import load_dotenv
from io import BytesIO
import pyarrow.parquet as pq

In [2]:
load_dotenv()

True

In [3]:
s3 = boto3.client("s3",
    region_name="eu-west-3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)

In [4]:
s3

In [5]:
# --- CSV: Read products.csv ---
response = s3.get_object(Bucket="kickz-empire-data", Key="raw/catalog/products.csv")
df = pd.read_csv(StringIO(response["Body"].read().decode("utf-8")))

In [6]:
print(df.shape)        # rows × columns

(228, 21)


In [7]:
print(df.dtypes)       # column types

product_id                  str
product_uuid                str
brand                       str
model_name                  str
colorway                    str
category                    str
subcategory                 str
slug                        str
display_name                str
price_usd               float64
_internal_cost_usd      float64
_supplier_id                str
_warehouse_location         str
_internal_cost_code         str
available_sizes_json        str
image_urls                  str
weight_grams              int64
is_active                  bool
is_hype_product            bool
created_at                  str
tags                        str
dtype: object


In [8]:
print(df.head())       # first rows

  product_id                          product_uuid   brand  \
0   KE-10023  d5fedc1b-03b3-42b9-8d5e-48a3469b6434    Nike   
1   KE-10093  c4e7be0d-bc56-43ff-b369-e2a6ea91dfc5  Adidas   
2   KE-10218  b7ef4692-68cf-44a9-95f4-2bc9231580cf  Adidas   
3   KE-10203  4bea0a64-e959-441b-9773-ab6d258162df    Puma   
4   KE-10228  72404ee3-2717-4dfd-9c8d-c5bda3afd3ee    Puma   

                   model_name     colorway     category subcategory  \
0            Air Jordan 1 Low   Shadow 2.0     sneakers    sneakers   
1                   Forum Low  Black/White     sneakers    sneakers   
2              Trefoil Beanie         Grey  accessories        caps   
3  Essentials Big Logo Hoodie   Shadow 2.0      hoodies     hoodies   
4              Essentials Cap        White  accessories        caps   

                                         slug  \
0            nike-air-jordan-1-low-shadow-2.0   
1                adidas-forum-low-black-white   
2                       adidas-trefoil-beanie   
3  p

In [9]:
print(df.describe())   # statistics

        price_usd  _internal_cost_usd  weight_grams
count  228.000000          228.000000    228.000000
mean   109.211316           48.501316    579.653509
std     61.089286           28.254575    319.077343
min     14.000000            6.390000     60.000000
25%     61.105000           24.475000    318.750000
50%    105.830000           45.280000    523.000000
75%    151.312500           64.690000    812.250000
max    262.630000          132.190000   1191.000000


In [10]:
# --- JSONL: Read reviews.jsonl ---
response = s3.get_object(Bucket="kickz-empire-data", Key="raw/reviews/reviews.jsonl")
jsonl_content = response["Body"].read().decode("utf-8")

# pd.read_json() with lines=True reads one JSON object per line
df_reviews = pd.read_json(StringIO(jsonl_content), lines=True)

In [11]:
print(df_reviews.shape)

(2930, 20)


In [12]:
print(df_reviews.dtypes)

review_id                             str
product_id                            str
product_name                          str
user_id                               str
user_name                             str
rating                              int64
title                                 str
body                                  str
verified_purchase                    bool
submitted_at          datetime64[us, UTC]
moderation_status                     str
moderated_at          datetime64[us, UTC]
helpful_votes                       int64
reported                             bool
photos_count                        int64
_moderation_score                 float64
_sentiment_raw                    float64
_toxicity_score                   float64
_language_detected                    str
_review_source                        str
dtype: object


In [13]:
print(df_reviews.head())

                              review_id product_id  \
0  16336ca4-ccfd-4089-999d-19cb0960f2a7   KE-10062   
1  46cfa7c2-7a8e-491f-b15f-91c52e936e42   KE-10141   
2  804cb048-dafd-4265-84be-50499a7ad9f8   KE-10160   
3  483504a3-ea06-4b77-9bb4-b3ac0b60b2d6   KE-10122   
4  359d1fc4-83f4-4b9a-88ce-603da1095ea0   KE-10216   

                                 product_name     user_id   user_name  rating  \
0             Nike ACG Dri-FIT Tee Georgetown  USR-011515    Chloe H.       5   
1     Jordan Air Jordan 11 Retro Court Purple  USR-010333     Zoey C.       4   
2                 New Balance 574 Lucky Green  USR-013884     Luke H.       4   
3  Adidas Adicolor Classics Tee Vintage Green  USR-014915  Abigail K.       5   
4                  Nike Heritage86 Futura Cap  USR-013935     Nora C.       4   

                       title  \
0                 Must have!   
1          Nice but runs big   
2       Good value for money   
3         Incredible quality   
4  Great overall, minor flaw

In [14]:
# --- Parquet: Read a single partition file ---
response = s3.get_object(
    Bucket="kickz-empire-data",
    Key="raw/clickstream/dt=2026-02-05/part-00001.snappy.parquet"
)
table = pq.read_table(BytesIO(response["Body"].read()))
df_click = table.to_pandas()

In [15]:
print(df_click.shape)

(5000, 27)


In [17]:
print(df_click.dtypes)

event_id                   str
event_type                 str
timestamp                  str
timestamp_epoch_ms       int64
user_id                    str
session_id                 str
page_url                   str
page_path                  str
page_type                  str
referrer_url               str
referrer_source            str
user_agent_raw             str
ip_address                 str
viewport_width           int64
viewport_height          int64
screen_color_depth       int64
device_pixel_ratio     float64
accept_language            str
is_bot                    bool
_ga_client_id              str
_gtm_container_id          str
_dom_interactive_ms      int64
_dom_complete_ms         int64
_ttfb_ms                 int64
_connection_type           str
_js_heap_size_mb       float64
_consent_string            str
dtype: object


In [18]:
print(df_click.head())

                               event_id event_type  \
0  ac0e758c-cc0b-4f45-b1c7-2f63dc1a0af1   pageview   
1  7d534b62-0fae-493c-897b-3ddc5d746fd9   pageview   
2  7a3a75e9-578b-4a6b-9097-15cb3a58436c   pageview   
3  54a88eb6-b33b-4e11-9d2f-9822b8d681fe   pageview   
4  f0c86354-3cdc-4519-b218-60e546b426ef   pageview   

                     timestamp  timestamp_epoch_ms     user_id  \
0  2026-02-05T14:46:58.590909Z       1770299218590  USR-012656   
1  2026-02-05T10:48:30.237093Z       1770284910237         NaN   
2  2026-02-05T00:51:17.162586Z       1770249077162  USR-010852   
3  2026-02-05T17:19:01.805154Z       1770308341805  USR-010617   
4  2026-02-05T15:21:30.548462Z       1770301290548  USR-012860   

         session_id                                      page_url  \
0  cdde1ca257a9458e           https://www.kickzempire.com/account   
1  8dbe6504380b4968  https://www.kickzempire.com/checkout/payment   
2  8f3e5a0aa69c40b8    https://www.kickzempire.com/search?q=yeezy   
3 